# 🤖 Notebook 05 – Model Training
**Goal:** train two baseline classifiers and compare them.

| Model | Idea |
|---|---|
| **Multinomial Naive Bayes** | Uses word probabilities per class and Bayes' theorem. Very fast, great text baseline. |
| **Logistic Regression** | Learns a weight for every word; the weights decide the class. Usually stronger and easy to interpret. |

## 1. Import Libraries

In [ ]:
# Import models, metrics and utilities
import time
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

In [ ]:
# Set up project paths (works on your D: drive path, or falls back to the parent of 02_Notebooks)
from pathlib import Path

BASE_DIR = Path(r"D:\DataScience\notebooks\SentimentIQ")
if not BASE_DIR.exists():
    BASE_DIR = Path.cwd().parent          # fallback: notebook is running inside 02_Notebooks/

RAW_PATH       = BASE_DIR / "01_Data" / "Raw" / "Tweets.csv"
PROCESSED_DIR  = BASE_DIR / "01_Data" / "Processed"
CLEANED_PATH   = PROCESSED_DIR / "cleaned_tweets.csv"
MODELS_DIR     = BASE_DIR / "03_Models"
REPORTS_DIR    = BASE_DIR / "05_Reports"

for folder in (PROCESSED_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42      # same seed everywhere -> reproducible results
TEST_SIZE    = 0.20    # 80% training / 20% testing

print("Project folder :", BASE_DIR)
print("Raw data exists:", RAW_PATH.exists())

## 2. Define X and y and Split the Dataset

In [ ]:
# Load the cleaned dataset and recreate the SAME train/test split that was made in Notebook 4
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    df = pd.read_csv(CLEANED_PATH)
except FileNotFoundError:
    raise FileNotFoundError("cleaned_tweets.csv not found. Please run Notebook 03 first.")

df = df.dropna(subset=["cleaned_text", "airline_sentiment"]).reset_index(drop=True)

X = df["cleaned_text"]          # input feature
y = df["airline_sentiment"]     # target variable

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print("Train size:", len(X_train), "| Test size:", len(X_test))

In [ ]:
# Safety check: the split must be identical to the one saved in Notebook 04
try:
    saved_train = pd.read_csv(PROCESSED_DIR / "train.csv")
    assert (saved_train["cleaned_text"].values == X_train.values).all()
    print("✅ Split matches Notebook 04 (train.csv).")
except FileNotFoundError:
    print("train.csv not found - skipping the consistency check.")
except AssertionError:
    print("⚠️ The split differs from Notebook 04. Re-run Notebooks 03 and 04, then this notebook.")

## 3. Load the TF-IDF Vectorizer and Create Features

In [ ]:
# Load the vectorizer fitted on the training data in Notebook 04, then transform both sets
try:
    tfidf = joblib.load(MODELS_DIR / "tfidf_vectorizer.pkl")
except FileNotFoundError:
    raise FileNotFoundError("tfidf_vectorizer.pkl not found. Please run Notebook 04 first.")

X_train_tfidf = tfidf.transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
print("Train features:", X_train_tfidf.shape, "| Test features:", X_test_tfidf.shape)

## 4. Train Model 1 – Multinomial Naive Bayes

In [ ]:
# Train Multinomial Naive Bayes (alpha = smoothing so unseen words do not get zero probability)
nb_model = MultinomialNB(alpha=1.0)

start = time.time()
nb_model.fit(X_train_tfidf, y_train)
nb_time = time.time() - start

print(f"✅ Naive Bayes trained in {nb_time:.2f} seconds")
print("Classes learned:", list(nb_model.classes_))

## 5. Train Model 2 – Logistic Regression

In [ ]:
# Train Logistic Regression.
# class_weight="balanced" gives the rare classes (neutral/positive) more importance, which helps with imbalance.
lr_model = LogisticRegression(
    max_iter=1000, C=1.0, class_weight="balanced", solver="lbfgs", random_state=RANDOM_STATE
)

start = time.time()
lr_model.fit(X_train_tfidf, y_train)
lr_time = time.time() - start

print(f"✅ Logistic Regression trained in {lr_time:.2f} seconds")
print("Classes learned:", list(lr_model.classes_))

## 6. Quick Performance Check (train vs test)

In [ ]:
# Compare training and testing scores. A big gap means overfitting (memorising instead of learning).
rows = []
for name, model, secs in [("Naive Bayes", nb_model, nb_time), ("Logistic Regression", lr_model, lr_time)]:
    train_pred = model.predict(X_train_tfidf)
    test_pred  = model.predict(X_test_tfidf)
    rows.append({
        "Model": name,
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Test Accuracy":  accuracy_score(y_test, test_pred),
        "Test Macro F1":  f1_score(y_test, test_pred, average="macro"),
        "Train-Test Gap": accuracy_score(y_train, train_pred) - accuracy_score(y_test, test_pred),
        "Training Time (s)": secs,
    })

baseline_results = pd.DataFrame(rows).set_index("Model").round(4)
baseline_results

**Expected output:** both models should land roughly between 0.75 and 0.82 test accuracy; Logistic Regression is normally ahead on macro F1 because Naive Bayes tends to over-predict the majority class (*negative*). The detailed analysis is in Notebook 06.

## 7. Cross-Validation (more reliable than one split)

In [ ]:
# 5-fold cross-validation on the TRAINING set only. A Pipeline refits TF-IDF inside every fold,
# so no information from the validation fold leaks into the vectorizer.
def make_pipeline(classifier):
    return Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
        ("clf", classifier),
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, clf in [("Naive Bayes", MultinomialNB(alpha=1.0)),
                  ("Logistic Regression", LogisticRegression(max_iter=1000, class_weight="balanced"))]:
    scores = cross_val_score(make_pipeline(clf), X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)
    cv_rows.append({"Model": name, "CV Macro F1 (mean)": scores.mean(), "CV Std": scores.std(),
                    "Fold scores": np.round(scores, 3).tolist()})

pd.DataFrame(cv_rows).set_index("Model").round(4)

## 8. Try the Models on Sample Sentences

In [ ]:
# Predict a few already-cleaned sentences with both models (full preprocessing + app logic comes in Notebook 08)
samples = ["great service love crew", "not happy delay terrible", "need change flight tomorrow"]
sample_tfidf = tfidf.transform(samples)

result = pd.DataFrame({
    "text": samples,
    "NB prediction": nb_model.predict(sample_tfidf),
    "NB confidence": nb_model.predict_proba(sample_tfidf).max(axis=1).round(3),
    "LR prediction": lr_model.predict(sample_tfidf),
    "LR confidence": lr_model.predict_proba(sample_tfidf).max(axis=1).round(3),
})
result

## 9. Save the Baseline Models

In [ ]:
# Save both baseline models; Notebook 06 will evaluate them in detail
try:
    joblib.dump(nb_model, MODELS_DIR / "baseline_naive_bayes.pkl")
    joblib.dump(lr_model, MODELS_DIR / "baseline_logistic_regression.pkl")
    baseline_results.to_csv(REPORTS_DIR / "baseline_training_summary.csv")
    print("✅ Models saved in", MODELS_DIR)
except Exception as e:
    print("❌ Saving failed:", e)

## ✅ Summary
- Two models were trained on the same TF-IDF features and compared on training vs. testing data.
- Cross-validation with a proper `Pipeline` gives a leakage-free estimate of performance.
- Baseline models are saved.

**Next:** Notebook 06 – Model Evaluation.